# Урок 24. Практикум ОГЭ: электронные таблицы

9 класс · III четверть

[⬆ Все уроки](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-09/index.ipynb) · [← Урок 23](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-09/urok-23.ipynb) · [Урок 25 →](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-09/urok-25.ipynb)

---

Разбор задания 14 ОГЭ. Работа с большим массивом данных в таблице. Проверка ответа независимым способом.

In [ ]:
#@title 🚀 Регистрация { display-mode: "form" }
#@markdown Впиши свои данные и запусти ячейку (Shift+Enter).
#@markdown Если заводил секрет `INFORMATIKA` и включил к нему доступ
#@markdown (🔑 на панели слева) — поля можно оставить пустыми.
#@markdown
#@markdown Colab спросит разрешение на доступ к аккаунту — нажми
#@markdown «Разрешить». Так проверка понимает, чья это работа.
ФИО = "" #@param {type:"string"}
#@markdown Класс — с буквой, например 9А
Класс = "" #@param {type:"string"}

import importlib, urllib.request
urllib.request.urlretrieve("https://raw.githubusercontent.com/LeksssGray/informatika-7-11/main/lib/schoolinf.py", "schoolinf.py")
import schoolinf as si
importlib.reload(si)
si.start(lesson="09-24", name=ФИО, klass=Класс)

## Разбираемся

### Что спрашивают в задании 14

Задание 14 ОГЭ — единственное, где вам дают **файл с данными**
и разрешают считать чем угодно: таблицей, программой, хоть
калькулятором. Обычно это выгрузка вроде такой:

| A | B | C | D | E |
|---|---|---|---|---|
| фамилия | класс | информатика | физика | математика |
| Иванов | 9А | 85 | 72 | 90 |
| Петрова | 9Б | 64 | 88 | 71 |

Строк в файле — от нескольких сотен до нескольких тысяч. Вопросов
обычно два, и они почти всегда одного из трёх видов:

1. **сколько** участников удовлетворяют условию;
2. **какой средний балл** у тех, кто удовлетворяет условию;
3. **сколько процентов** составляют такие участники от общего числа.

Оба ответа записываются целыми числами (или с указанной точностью)
в соседние ячейки — например, в H2 и H3.

### Три приёма, которые закрывают почти всё

**Приём 1. Вспомогательный столбец.** Вместо того чтобы городить
страшную формулу, заводят отдельный столбец с пометкой «подходит /
не подходит»:

```
F2:  =ЕСЛИ(И(C2>75; D2>75); 1; 0)
```

Дальше ответ — это просто `=СУММ(F2:F1273)`. Проверять такую
конструкцию легко: видно, какие строки отмечены.

**Приём 2. Условные функции.** Когда условие простое, хватает
одной формулы:

```
=СЧЁТЕСЛИ(B2:B1273; "9А")
=СРЗНАЧЕСЛИ(B2:B1273; "9А"; C2:C1273)
=СЧЁТЕСЛИМН(C2:C1273; ">75"; D2:D1273; ">75")
```

**Приём 3. Проверка вторым способом.** Тот же ответ считают ещё раз
иначе: формулой другого вида или программой. Совпало — сдаём.
Не совпало — ищем ошибку, и почти всегда она в границах диапазона.

### Где теряют баллы

| Ошибка | Как выглядит |
|---|---|
| Диапазон не до конца данных | `C2:C1000` вместо `C2:C1273` |
| Захватили строку заголовков | `C1:C1273` — в среднее попал текст |
| Среднее поделили на всех | делить надо на количество подходящих |
| Округлили не там | сначала считают, округляют в самом конце |
| Ответ записали не в ту ячейку | проверьте условие: H2 или H3 |

Первая строка — чемпион по потерянным баллам. Поэтому диапазон
выделяют не мышкой, а клавишами `Ctrl` + `Shift` + `↓`: они дойдут
ровно до конца данных.

### Точность

«Ответ округлите до двух знаков» означает **округлить один раз,
в самом конце**. Если округлить каждое слагаемое, а потом сложить,
набежит расхождение — и ответ не засчитают.

В Python это `round(значение, 2)` — и тоже только в конце.

## Смотрим, как это работает

Разберём настоящее задание целиком: сгенерируем файл, решим двумя
способами и сверим.

### Пример 1. Файл с данными

In [ ]:
import random

random.seed(14)

КЛАССЫ = ["9А", "9Б", "9В"]
таблица = []

for номер in range(1, 1273):
    таблица.append([
        f"Участник{номер:04d}",
        random.choice(КЛАССЫ),
        random.randint(40, 100),     # информатика
        random.randint(40, 100),     # физика
        random.randint(40, 100),     # математика
    ])

print("Строк данных:", len(таблица))
print("В таблице они занимают строки со 2 по", len(таблица) + 1)
print()
for строка in таблица[:3]:
    print(строка)

Обратите внимание на вторую строчку вывода: данных 1272, а последняя
строка таблицы — 1273, потому что первую занимает заголовок.
Именно здесь чаще всего и ошибаются с диапазоном.

### Пример 2. Вопрос первый — сколько

«Сколько участников набрали больше 75 баллов и по информатике,
и по физике?»

In [ ]:
подходят = 0
for фамилия, класс, инф, физ, мат in таблица:
    if инф > 75 and физ > 75:
        подходят += 1

print("Ответ 1:", подходят)
print()
print("В таблице это:")
print('  =СЧЁТЕСЛИМН(C2:C1273; ">75"; D2:D1273; ">75")')
print("  или вспомогательный столбец с ЕСЛИ и СУММ по нему.")

### Пример 3. Вопрос второй — средний балл

«Каков средний балл по математике среди участников, набравших
больше 75 по информатике? Ответ округлите до двух знаков.»

In [ ]:
сумма = 0
сколько = 0

for фамилия, класс, инф, физ, мат in таблица:
    if инф > 75:
        сумма += мат
        сколько += 1

средний = сумма / сколько

print("Подходящих участников:", сколько)
print("Ответ 2:", round(средний, 2))
print()
print("В таблице: =СРЗНАЧЕСЛИ(C2:C1273; \">75\"; E2:E1273)")

Делили на `сколько`, а не на 1272 — это второе классическое место
потери балла.

### Пример 4. Проверка вторым способом

Посчитаем тот же средний балл иначе: сначала отберём подходящие
значения в список, потом усредним встроенными функциями.

In [ ]:
отобранные = [мат for фамилия, класс, инф, физ, мат in таблица if инф > 75]

второй_способ = sum(отобранные) / len(отобранные)

print("Первый способ: ", round(средний, 2))
print("Второй способ: ", round(второй_способ, 2))
print("Совпало:", abs(средний - второй_способ) < 1e-9)

### Пример 5. Чем опасен неверный диапазон

Посмотрим, что будет, если в формуле указать `C2:C1000` вместо
`C2:C1273` — то есть потерять хвост данных.

In [ ]:
усечённая = таблица[:999]

неверно = 0
for фамилия, класс, инф, физ, мат in усечённая:
    if инф > 75 and физ > 75:
        неверно += 1

print("Правильный ответ:", подходят)
print("С усечённым диапазоном:", неверно)
print("Разница:", подходят - неверно, "участников")

Ответ выглядит совершенно правдоподобно — и потому такую ошибку
невозможно заметить, не проверив диапазон специально.

### Пример 6. Проценты

In [ ]:
из_9а = 0
for фамилия, класс, инф, физ, мат in таблица:
    if класс == "9А":
        из_9а += 1

доля = из_9а / len(таблица) * 100

print(f"Участников из 9А: {из_9а} из {len(таблица)}")
print(f"Это {round(доля, 2)} %")

## Пробуем сами

В задачах таблица приходит списком строк вида
`[фамилия, класс, информатика, физика, математика]`.

### Задача 1. Сколько строк данных

Данные занимают строки со 2 по 1273 включительно (первая строка —
заголовки). Сколько участников в таблице?

In [ ]:
#@title 🧩 Задача 1. Сколько участников { display-mode: "form" }
#@markdown Впиши число
участников = 0 #@param {type:"integer"}

si.ответ("1", участников, "c676b6e9a4ca6b5a",
         hint="Обе границы входят, но заголовок не считается.")

### Задача 2. Количество по двум условиям

Функция возвращает количество участников, набравших больше `порог`
и по информатике, и по физике.

In [ ]:
def сколько_сильных(таблица, порог):
    return ...

In [ ]:
si.check("2", сколько_сильных, [
    (([["А", "9А", 80, 80, 50], ["Б", "9Б", 80, 70, 90]], 75), 1),
    (([["А", "9А", 80, 80, 50]], 85), 0),
    (([["А", "9А", 90, 90, 90], ["Б", "9Б", 95, 95, 95]], 75), 2),
    (([], 50), 0),
])

### Задача 3. Средний балл среди подходящих

Функция возвращает средний балл по математике среди участников,
набравших больше `порог` по информатике. Ответ округлите
до двух знаков.

Если подходящих нет, верните 0.

In [ ]:
def средний_по_математике(таблица, порог):
    return ...

In [ ]:
si.check("3", средний_по_математике, [
    (([["А", "9А", 80, 70, 90], ["Б", "9Б", 60, 70, 50]], 75), 90.0),
    (([["А", "9А", 80, 70, 90], ["Б", "9Б", 90, 70, 71]], 75), 80.5),
    (([["А", "9А", 50, 70, 90]], 75), 0),
    (([], 75), 0),
])

### Задача 4. Сравнение двух предметов

Функция возвращает количество участников, у которых балл по физике
**строго больше** балла по математике.

In [ ]:
def физика_сильнее(таблица):
    return ...

In [ ]:
si.check("4", физика_сильнее, [
    ([["А", "9А", 50, 80, 70], ["Б", "9Б", 50, 60, 90]], 1),
    ([["А", "9А", 50, 70, 70]], 0),
    ([["А", "9А", 50, 99, 1], ["Б", "9Б", 50, 99, 1]], 2),
    ([], 0),
])

### Задача 5. Доля в процентах

Функция возвращает, сколько процентов участников учатся в заданном
классе. Ответ округлите до двух знаков.

Если таблица пуста, верните 0.

In [ ]:
def доля_класса(таблица, класс):
    return ...

In [ ]:
si.check("5", доля_класса, [
    (([["А", "9А", 1, 1, 1], ["Б", "9Б", 1, 1, 1],
       ["В", "9А", 1, 1, 1], ["Г", "9В", 1, 1, 1]], "9А"), 50.0),
    (([["А", "9А", 1, 1, 1], ["Б", "9Б", 1, 1, 1], ["В", "9Б", 1, 1, 1]], "9Б"), 66.67),
    (([["А", "9А", 1, 1, 1]], "9Я"), 0.0),
    (([], "9А"), 0),
])

### Задача 6. Функция для количества

Какой функцией в таблице считают количество ячеек, подходящих
под одно условие? Запишите заглавными буквами, по-русски.

In [ ]:
#@title 🧩 Задача 6. Функция подсчёта { display-mode: "form" }
#@markdown Впиши название функции
функция = "" #@param {type:"string"}

si.ответ("6", функция, "c8cf17da3496edbe",
         hint="«Сосчитай, если…»")

## Домашнее задание

### Домашнее задание 1. Четыре задачи формата ОГЭ

Возьмите любой файл с данными хотя бы на 500 строк (подойдёт файл
с сайта ФИПИ, выгрузка из журнала или таблица из открытых данных)
и ответьте на четыре вопроса:

1. сколько записей удовлетворяют условию по двум столбцам сразу;
2. каково среднее по третьему столбцу среди них (два знака);
3. сколько это процентов от общего числа записей;
4. какое максимальное значение среди подходящих.

Каждый ответ посчитайте **дважды**: формулой и программой.
Принесите оба результата — и, если они разошлись, объяснение почему.

### Домашнее задание 2. Максимум среди подходящих

Функция возвращает наибольший балл по информатике среди участников
заданного класса. Если таких нет, верните 0.

In [ ]:
def лучший_в_классе(таблица, класс):
    return ...

In [ ]:
si.check("дз2", лучший_в_классе, [
    (([["А", "9А", 80, 1, 1], ["Б", "9Б", 95, 1, 1], ["В", "9А", 88, 1, 1]], "9А"), 88),
    (([["А", "9А", 80, 1, 1]], "9Б"), 0),
    (([["А", "9А", 40, 1, 1]], "9А"), 40),
    (([], "9А"), 0),
])

### Домашнее задание 3. Функция для среднего

Какой функцией в таблице считают среднее по значениям,
подходящим под условие? Запишите заглавными буквами, по-русски.

In [ ]:
#@title 🏠 Домашнее 3. Функция среднего { display-mode: "form" }
#@markdown Впиши название функции
функция_среднего = "" #@param {type:"string"}

si.ответ("дз3", функция_среднего, "8eb9bc087b143891",
         hint="«Среднее значение, если…»")

---

### Любопытно

Задание 14 — единственное в ОГЭ, где можно решить задачу тремя
совершенно разными способами и получить одинаковый балл: формулами,
программой или даже вручную, если хватит терпения.

Опыт показывает, что быстрее всего работают те, кто умеет оба
способа и выбирает по задаче: простое условие — формула за минуту,
хитрое («у скольких сумма трёх предметов больше, чем у соседа
по списку») — двадцать строк кода, которые в таблице пришлось бы
собирать полчаса.

---

## Отчёт по уроку

Запусти ячейку ниже, когда решишь задачи. Результат уйдёт учителю автоматически.

In [ ]:
si.report()

---

[← Урок 23](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-09/urok-23.ipynb) · [⬆ Все уроки](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-09/index.ipynb) · [🏠 Ко всем классам](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/index.ipynb) · [Урок 25 →](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-09/urok-25.ipynb)